In [21]:
import numpy as np
import matplotlib.pyplot as plt
import fastf1
import pandas as pd

In [22]:
# get drs data from fastf1 api, visualise it for exploration purposes.
# set the position for the cache
fastf1.Cache.enable_cache('../fastf1_cache')

In [23]:
# example in a qualifying session 


In [ ]:
# example using the same data used by Abhay in discreteLapbyLapResetTyre.py
YEAR = 2023
GP = "Spain"
SESSION = "R"
DRIVER = "VER"

def build_driver_car_data(year, gp, session_name, driver):
    session = fastf1.get_session(year, gp, session_name)
    session.load()

    laps = session.laps.pick_drivers(driver)
    lap_car_data = []

    for _, lap in laps.iterlaps():
        car_data = lap.get_car_data()
        car_data['LapNumber'] = lap['LapNumber']
        lap_car_data.append(car_data)

    car_data_df = pd.concat(lap_car_data, ignore_index=True)
    return session, laps, lap_car_data, car_data_df


def visualise_drs_data(
    car_data_df,
    driver,
    gp,
    year,
    session_name,
    lap_numbers=None,
    drop_first_n_laps=2,
    fig_size=(10, 6),
):
    """Visualise DRS usage by lap average and per-lap traces."""
    print(car_data_df.head())

    drs_activation_by_lap = car_data_df.groupby('LapNumber')['DRS'].mean()
    if drop_first_n_laps > 0:
        drs_activation_by_lap = drs_activation_by_lap[drop_first_n_laps:]

    plt.figure(figsize=fig_size)
    plt.plot(drs_activation_by_lap.index, drs_activation_by_lap.values, marker='o')
    plt.xlabel('Lap Number')
    plt.ylabel('Mean DRS Activation')
    plt.title(f'DRS Activation for {driver} in {gp} {year} {session_name}')
    plt.grid()
    plt.show()

    if lap_numbers is None:
        lap_numbers = sorted(car_data_df['LapNumber'].dropna().unique())

    plt.figure(figsize=fig_size)
    for lap_number in lap_numbers:
        lap_data = car_data_df[car_data_df['LapNumber'] == lap_number].reset_index(drop=True)
        if lap_data.empty:
            continue
        plt.plot(lap_data.index, lap_data['DRS'], label=f'Lap {int(lap_number)}')

    plt.legend()
    plt.xlabel('Index of the data point in the lap')
    plt.ylabel('DRS Activated (1 = Yes, 0 = No)')
    plt.title(f'DRS Activation for {driver} in {gp} {year} {session_name}')
    plt.grid()
    plt.show()

    return drs_activation_by_lap


session, laps, lap_car_data, car_data_df = build_driver_car_data(YEAR, GP, SESSION, DRIVER)

core           INFO 	Loading data for Spanish Grand Prix - Race [v3.8.1]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core        WARNING 	Driver 1 completed the race distance 00:00.037000 before the recorded end of the session.
core           INFO 	Finished loading data for 20 drivers: ['1', '44', '63', '11', '55', '18', '14', '31', '24', '10', '16', '2

In [ ]:
# call the reusable DRS visualisation function
drs_activation_by_lap = visualise_drs_data(
    car_data_df=car_data_df,
    driver=DRIVER,
    gp=GP,
    year=YEAR,
    session_name=SESSION,
    lap_numbers=[3, 4, 5, 6, 7, 8, 9, 10],
    drop_first_n_laps=2,
)

                     Date      RPM  Speed  nGear  Throttle  Brake  DRS Source  \
0 2023-06-04 13:03:16.150  10072.0    0.0      1      16.0   True    1    car   
1 2023-06-04 13:03:16.550   8862.0    5.0      1      17.0  False    1    car   
2 2023-06-04 13:03:16.990   5782.0   17.0      1      18.0  False    1    car   
3 2023-06-04 13:03:17.230   4600.0   26.0      1      21.0  False    1    car   
4 2023-06-04 13:03:17.510   4266.0   37.0      1      30.0  False    1    car   

                    Time            SessionTime  LapNumber  
0 0 days 00:00:00.180000 0 days 01:02:16.143000        1.0  
1 0 days 00:00:00.580000 0 days 01:02:16.543000        1.0  
2 0 days 00:00:01.020000 0 days 01:02:16.983000        1.0  
3 0 days 00:00:01.260000 0 days 01:02:17.223000        1.0  
4 0 days 00:00:01.540000 0 days 01:02:17.503000        1.0  


/var/folders/63/__1p4tt55zq_byscd0qttqcc0000gn/T/ipykernel_63197/2526829942.py:5: FutureWarning: The behavior of obj[i:j] with a float-dtype index is deprecated. In a future version, this will be treated as positional instead of label-based. For label-based slicing, use obj.loc[i:j] instead
  drs_activation_by_lap = drs_activation_by_lap[2:]
